In [16]:

from pyspark.sql import *
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("Spark Optimization") \
.config("spark.sql.ui.explainMode", "extended").getOrCreate()

In [17]:
df = spark.read.format("csv").option("header", True).option("inferSchema", True)\
.load("BigMart Sales.csv")
df.show(5)

+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|Item_Identifier|Item_Weight|Item_Fat_Content|Item_Visibility|           Item_Type|Item_MRP|Outlet_Identifier|Outlet_Establishment_Year|Outlet_Size|Outlet_Location_Type|      Outlet_Type|Item_Outlet_Sales|
+---------------+-----------+----------------+---------------+--------------------+--------+-----------------+-------------------------+-----------+--------------------+-----------------+-----------------+
|          FDA15|        9.3|         Low Fat|    0.016047301|               Dairy|249.8092|           OUT049|                     1999|     Medium|              Tier 1|Supermarket Type1|         3735.138|
|          DRC01|       5.92|         Regular|    0.019278216|         Soft Drinks| 48.2692|           OUT018|                     2009|     Medium|              Tier 3|Superma

In [9]:
df.select("Outlet_Type").distinct().show()

+-----------------+
|      Outlet_Type|
+-----------------+
|Supermarket Type3|
|    Grocery Store|
|Supermarket Type2|
|Supermarket Type1|
+-----------------+



# partitioned data 

In [10]:
df.write.format("parquet").mode("append")\
.partitionBy("Outlet_Type")\
.option("path", "outputData/partitioned-Outlet_Type").save()

# non partitioned Data 

In [11]:
df.write.format("parquet").mode("append")\
.option("path", "outputData/non-partitioned").save()

# read data

In [12]:
df1 = spark.read.format("parquet")\
.option("path", "outputData/partitioned-Outlet_Type").load()
df1.rdd.getNumPartitions()

4

In [13]:
df2 = spark.read.format("parquet")\
.option("path", "outputData/non-partitioned").load()
df2.rdd.getNumPartitions()

1

# set up spark config

In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "true") # Enable AQE
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false") # Disable DPP
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # Disable broadcast joins

# JOIN

In [18]:
df_join = df1.join(df2.filter(col("Outlet_Type") == "Grocery Store"), df1["Item_Identifier"] == df2["Item_Identifier"], "inner" )
# df_join.explain(extended=True)
df_join.count()

6621